# NLP Project Pipeline
## Step-by-Step Guide for Building an NLP Project

**Project:** Sentiment/Text Classification Pipeline

This notebook demonstrates an end-to-end NLP engineering workflow:

`Problem → Data → EDA → Cleaning → TF-IDF → Train/Test Split → Model → Evaluation → Error Analysis → Save → Inference`

> This is a beginner-friendly baseline. For a production project, replace the small example dataset with a larger, properly labeled dataset.

## 1. Install / Import Libraries

Run the following cell if the libraries are not installed.

In [ ]:
# If required, uncomment and run:
# %pip install pandas matplotlib scikit-learn joblib

import re
import html
import joblib
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully.")

## 2. Problem Definition

We will build a binary text-classification model.

**Input:** A short review  
**Output:** `positive` or `negative`

This is a **supervised learning** problem because each training text has a known label.

## 3. Create Example Dataset

A small dataset is included so the notebook can run without downloading external files.

In [ ]:
data = [
    ("I absolutely love this product", "positive"),
    ("The product is excellent and useful", "positive"),
    ("Amazing experience and great quality", "positive"),
    ("I am very happy with the purchase", "positive"),
    ("This is a fantastic product", "positive"),
    ("The service was quick and helpful", "positive"),
    ("The quality is outstanding", "positive"),
    ("I would definitely recommend it", "positive"),
    ("Very good experience overall", "positive"),
    ("The product works perfectly", "positive"),
    ("I enjoyed using this product", "positive"),
    ("Great value and excellent performance", "positive"),
    ("The support team was friendly", "positive"),
    ("Fast delivery and good packaging", "positive"),
    ("This is one of my favorite products", "positive"),
    ("The product is reliable and simple", "positive"),
    ("I am satisfied with the results", "positive"),
    ("Wonderful service from start to finish", "positive"),
    ("The experience was pleasant", "positive"),
    ("Good quality for the price", "positive"),
    ("I hate this product", "negative"),
    ("The product is terrible", "negative"),
    ("Very bad experience", "negative"),
    ("I am disappointed with the purchase", "negative"),
    ("This product is useless", "negative"),
    ("The service was slow and unhelpful", "negative"),
    ("The quality is extremely poor", "negative"),
    ("I would not recommend this product", "negative"),
    ("The experience was awful", "negative"),
    ("The product stopped working", "negative"),
    ("I regret buying this", "negative"),
    ("Poor quality and bad performance", "negative"),
    ("The support team was rude", "negative"),
    ("Delivery was late and packaging was damaged", "negative"),
    ("This is one of the worst products", "negative"),
    ("The product is unreliable", "negative"),
    ("I am unhappy with the results", "negative"),
    ("Terrible service from start to finish", "negative"),
    ("The experience was frustrating", "negative"),
    ("Bad quality for the price", "negative"),
]

df = pd.DataFrame(data, columns=["text", "label"])

print("Dataset shape:", df.shape)
df.head()

## 4. Data Understanding / EDA

In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nClass distribution:")
print(df["label"].value_counts())

df["text_length"] = df["text"].str.len()

print("\nText length statistics:")
print(df["text_length"].describe())

In [ ]:
df["label"].value_counts().plot(kind="bar")
plt.title("Class Distribution")
plt.xlabel("Label")
plt.ylabel("Number of Samples")
plt.xticks(rotation=0)
plt.show()

## 5. Text Cleaning

We will:
- Convert text to lowercase
- Decode HTML entities
- Remove URLs
- Remove HTML tags
- Remove punctuation
- Normalize whitespace

The cleaning function is deliberately simple and should be adapted to the real dataset.

In [ ]:
def clean_text(text):
    text = html.unescape(str(text))
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_text"] = df["text"].apply(clean_text)

df[["text", "clean_text", "label"]].head()

## 6. Train/Test Split

We split the data **before fitting the TF-IDF vectorizer**.

This is important because the test set must remain unseen during training.

`stratify=y` keeps the class proportions approximately consistent between the two splits.

In [ ]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

## 7. Feature Extraction with TF-IDF

TF-IDF converts text into numerical features.

We use:
- `ngram_range=(1, 2)` → unigrams + bigrams
- `min_df=1` → keep terms appearing in at least one training document
- `sublinear_tf=True` → use a logarithmic TF scaling

The vectorizer is placed inside the model pipeline so it is fitted only on training data.

## 8. Build the NLP Model Pipeline

Architecture:

```text
Cleaned Text
     ↓
TF-IDF Vectorizer
     ↓
Logistic Regression
     ↓
Prediction
```

In [ ]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            ngram_range=(1, 2),
            min_df=1,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    )
])

print(model)

## 9. Train the Model

In [ ]:
model.fit(X_train, y_train)

print("Model training completed.")

## 10. Evaluate the Model

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 11. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=["negative", "positive"])

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["negative", "positive"]
)

disp.plot()
plt.title("Confusion Matrix")
plt.show()

## 12. Error Analysis

Looking at wrong predictions is important because metrics alone do not explain model behavior.

We will display the test examples that were classified incorrectly.

In [ ]:
error_df = pd.DataFrame({
    "text": X_test,
    "actual": y_test,
    "predicted": y_pred
})

errors = error_df[error_df["actual"] != error_df["predicted"]]

print("Number of errors:", len(errors))
errors.reset_index(drop=True)

## 13. Inspect Important Features

For Logistic Regression with TF-IDF, the learned coefficients can help us inspect terms associated with each class.

Positive coefficients push predictions toward the positive class; negative coefficients push them toward the negative class.

In [ ]:
vectorizer = model.named_steps["tfidf"]
classifier = model.named_steps["classifier"]

feature_names = vectorizer.get_feature_names_out()
coefficients = classifier.coef_[0]

feature_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefficients
})

print("Features associated with positive class:")
print(feature_df.sort_values("coefficient", ascending=False).head(10))

print("\nFeatures associated with negative class:")
print(feature_df.sort_values("coefficient", ascending=True).head(10))

## 14. Save the Complete NLP Pipeline

In [ ]:
model_path = "nlp_text_classifier.joblib"

joblib.dump(model, model_path)

print(f"Model saved to: {model_path}")

## 15. Load the Saved Model

In [ ]:
loaded_model = joblib.load(model_path)

print("Saved model loaded successfully.")

## 16. Inference on New Text

In [ ]:
new_texts = [
    "The product quality is amazing",
    "I am very disappointed with this service",
    "This is a great and useful product",
    "The experience was terrible"
]

predictions = loaded_model.predict(new_texts)

results = pd.DataFrame({
    "text": new_texts,
    "prediction": predictions
})

results

## 17. Optional: Prediction Probabilities

For classifiers that support probabilities, we can inspect the model's confidence-like probability estimates.

> These probabilities should not automatically be interpreted as perfectly calibrated real-world probabilities.

In [ ]:
probabilities = loaded_model.predict_proba(new_texts)

probability_df = pd.DataFrame(
    probabilities,
    columns=loaded_model.classes_
)

probability_df.insert(0, "text", new_texts)
probability_df

# 18. Complete NLP Pipeline Summary

```text
1. Problem Definition
2. Data Collection
3. Data Understanding / EDA
4. Text Cleaning
5. Train/Test Split
6. TF-IDF Feature Extraction
7. Logistic Regression
8. Model Training
9. Evaluation
10. Confusion Matrix
11. Error Analysis
12. Feature Inspection
13. Model Saving
14. Inference
15. Deployment
16. Monitoring
```

## Next Steps for a Real NLP Engineering Project

- Use a larger dataset.
- Add proper validation data or cross-validation.
- Compare Logistic Regression, Naive Bayes and Linear SVM.
- Experiment with word and character n-grams.
- Handle class imbalance if present.
- Perform deeper error analysis.
- Add experiment tracking.
- Create a FastAPI or Streamlit application.
- Containerize the application with Docker.
- Monitor model performance and data drift.
- For advanced NLP, experiment with transformer models such as BERT or DistilBERT.